In [87]:
import pandas as pd
import sqlite3
import numpy as np

Summary: Pandas, SQLAlchemy & SQL dialect
>sqlite3: Python module specifically for SQLite databases.

>>SQLAlchemy: Supports multiple databases using appropriate drivers.

>create_engine(): Determines the database dialect and driver from the connection URL.

>df.to_sql(): Inserts a Pandas DataFrame into a database table using the SQLAlchemy engine.

In [2]:
conn=sqlite3.connect('sales_customer.db')

In [4]:
# check tables in db
tables=pd.read_sql("select name from sqlite_master where type='table'",conn)
tables
# tables is df and index are not counted as columns in df unlike numpy array you cant iterate df of single column


,name
0,olist_customers_dataset
1,olist_geolocation_dataset
2,olist_orders_dataset
3,olist_order_items_dataset
4,olist_order_payments_dataset
5,olist_order_reviews_dataset
6,olist_products_dataset
7,olist_sellers_dataset
8,product_category_name_translation


In [5]:
df={}
for table in tables.iloc[:,0]:
    df[table]=(pd.read_sql_query(f"select * from {table}",conn))


df.head()

View first 5 rows

df.tail()

View last 5 rows

df.shape

Number of rows and columns

df.info()

Data types and non-null counts

df.describe()

Statistical summary

df.nunique()

Count unique values

df.isnull().sum()

Find missing values

df.duplicated().sum()

Find duplicate rows

df.value_counts()

Frequency of values

df.columns

Display column names

In [6]:
df_customers=df['olist_customers_dataset']

In [7]:
print(df_customers.isnull().sum(),"\n")
print(df_customers.info(),"\n")
print(df_customers.nunique(),"\n")


customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  object
 1   customer_unique_id        99441 non-null  object
 2   customer_zip_code_prefix  99441 non-null  int64 
 3   customer_city             99441 non-null  object
 4   customer_state            99441 non-null  object
dtypes: int64(1), object(4)
memory usage: 3.8+ MB
None 

customer_id                 99441
customer_unique_id          96096
customer_zip_code_prefix    14994
customer_city                4119
customer_state                 27
dtype: int64 



In [8]:
print(df_customers['customer_id'].duplicated().sum(),"\n")
print(df_customers.duplicated().sum(),"\n")

0 

0 



In [9]:
df_customers.groupby('customer_unique_id')['customer_state'].nunique().sort_values(ascending=False).value_counts()

customer_state
1    96057
2       38
3        1
Name: count, dtype: int64

In [10]:
uniq_cust_st=df_customers.groupby('customer_state')['customer_unique_id'].nunique().sort_values(ascending=False)
print(uniq_cust_st)
# print(uniq_cust_st.sum())
top_15=(uniq_cust_st.head(15).sum())/(df_customers['customer_unique_id'].nunique())
top_15

customer_state
SP    40302
RJ    12384
MG    11259
RS     5277
PR     4882
SC     3534
BA     3277
DF     2075
ES     1964
GO     1952
PE     1609
CE     1313
PA      949
MT      876
MA      726
MS      694
PB      519
PI      482
RN      474
AL      401
SE      342
TO      273
RO      240
AM      143
AC       77
AP       67
RR       45
Name: customer_unique_id, dtype: int64


0.9613199300699301

In [11]:
# multi_city_cust=df_customers.groupby('customer_unique_id')['customer_city'].nunique()
# multi_city_cust.sort_values().value_counts()

>no null values

>only zip code is int rest are object 

>more customer ids than customer unique ids

> top 5 cities account for 77.1%, top 10 cities for 90.43%,top 15 cities for 96.13% of customers


>more than 122 unique customer have records in 2 or more cities,rest are in only 1 state



In [12]:
# df_geoloc.groupby('geolocation_state')['geolocation_city'].nunique().sort_values(ascending=False)
# df['olist_geolocation_dataset'].groupby('geolocation_zip_code_prefix').size().sort_values(ascending=False).head(10)
df_geoloc=df['olist_geolocation_dataset']

In [13]:
print(df_geoloc.info(),"\n")
print("total duplicate rows=",df_geoloc.duplicated().sum(),"\n")
print("total duplicate cordinates=",df_geoloc[['geolocation_lat','geolocation_lng']].duplicated().sum(),"\n")
print("unique values in each col=","\n",df_geoloc.nunique(),"\n")
print("total zip code with associated states count=",df_geoloc.groupby('geolocation_zip_code_prefix')['geolocation_state'].nunique().value_counts(),"\n")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_zip_code_prefix  1000163 non-null  int64  
 1   geolocation_lat              1000163 non-null  float64
 2   geolocation_lng              1000163 non-null  float64
 3   geolocation_city             1000163 non-null  object 
 4   geolocation_state            1000163 non-null  object 
dtypes: float64(2), int64(1), object(2)
memory usage: 38.2+ MB
None 

total duplicate rows= 261831 

total duplicate cordinates= 281700 

unique values in each col= 
 geolocation_zip_code_prefix     19015
geolocation_lat                717360
geolocation_lng                717613
geolocation_city                 8011
geolocation_state                  27
dtype: int64 

total zip code with associated states count= geolocation_state
1    19007
2        8
Name: count

In [37]:
multiple_city_zip=(df_geoloc.groupby('geolocation_zip_code_prefix')['geolocation_city'].nunique())
mcz=pd.DataFrame(multiple_city_zip)
mcz.value_counts()

geolocation_city
1                   10459
2                    8265
3                     255
4                      27
5                       9
Name: count, dtype: int64


>total 1000163 rows across 5 columns more rows than customer data table

>zip code is int, co-ordinates are float, city and state are objects and all are non null

>total duplicate rows= 261831 , total duplicate cordinates= 281700, could imply same city/states have diff spelling or zip code

>unique zip codes are 19015 almost double the no of unique cities

>8 zip codes are associated with 2 states


In [15]:
multiple_state_zip=(df_geoloc.groupby('geolocation_zip_code_prefix')['geolocation_state'].nunique())
# df_[multiple_state_zip[multiple_state_zip==2]]

In [ ]:
# this is 3rd table
df_ord_itm=df['olist_order_items_dataset']

In [17]:
print(df_ord_itm.info(),"\n")
print(df_ord_itm.nunique(),"\n")
print(df_ord_itm.duplicated().sum(),"\n")
print(df_ord_itm.duplicated(subset=['order_id','order_item_id']).sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  object 
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  object 
 3   seller_id            112650 non-null  object 
 4   shipping_limit_date  112650 non-null  object 
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), object(4)
memory usage: 6.0+ MB
None 

order_id               98666
order_item_id             21
product_id             32951
seller_id               3095
shipping_limit_date    93318
price                   5968
freight_value           6999
dtype: int64 

0 

0


In [41]:
df_ord_itm.groupby('order_item_id')['order_id'].count()

order_item_id
1     98666
2      9803
3      2287
4       965
5       460
6       256
7        58
8        36
9        28
10       25
11       17
12       13
13        8
14        7
15        5
16        3
17        3
18        3
19        3
20        3
21        1
Name: order_id, dtype: int64

In [ ]:
df['shipping_date']=df_ord_itm['shipping_limit_date'].str[:10]

In [128]:
df['shipping_date'].str[5:7].value_counts()

shipping_limit_date
08    13857
05    12915
03    11510
07    10788
06    10698
04    10003
02     9243
01     8173
12     7727
11     7355
10     5554
09     4827
Name: count, dtype: int64

In [121]:
df['shipping_date'].str[:4].value_counts()

shipping_limit_date
2018    62511
2017    49765
2016      370
2020        4
Name: count, dtype: int64

In [81]:
df_ord_itm['product_id'].value_counts().loc[lambda x: x>100].count()
# 18117/32951
# 57/32951

57

In [99]:
17117/112650

0.1519485130936529

112650

In [97]:
df_ord_itm['product_id'].value_counts().loc[lambda x: x>1].sum()


94533

>total orders placed = 98666

>out of these only 9803 order have more than 1 item , so 90% orders are for single item

>we need analyze customer feedback from both sides

>out of 32951 unique product id 18117 were ordered only once and 57 orders were ordered more than 100 times making them 54% and 0.17% respectively

>those 54% of product id make for 15% of order samples


In [20]:
df_ord_itm.groupby('order_item_id')['order_id']

In [21]:
df_ord_itm['shipping_limit_date'].apply(lambda col :col[:4]).value_counts()

shipping_limit_date
2018    62511
2017    49765
2016      370
2020        4
Name: count, dtype: int64

In [22]:
df_ord_itm['shipping_limit_date'].apply(lambda col :col[:4]).nunique()

4

In [23]:
(df_ord_itm.groupby('order_id')['seller_id'].nunique().sort_values(ascending=False))

order_id
1c11d0f4353b31ac3417fbfa5f0f2a8a    5
cf5c8d9f52807cb2d2f0a0ff54c478da    5
8c2b13adf3f377c8f2b06b04321b0925    4
91be51c856a90d7efe86cf9d082d6ae3    4
1d23106803c48c391366ff224513fb7f    4
                                   ..
55e186387dc4b33d16d86a920ee7f77b    1
55e144afb3e4115f6c520fb6242cd1bf    1
55dfff244bc5d5f81d58d4148062c9b6    1
55dfe839c1db3060005158c694bddb4e    1
fffe41c64501cc87c801fd61db3f6244    1
Name: seller_id, Length: 98666, dtype: int64

In [24]:
print((df_ord_itm.groupby('seller_id')['order_item_id'].count().sort_values()).value_counts())
(df_ord_itm.groupby('seller_id')['order_id'].count().sort_values()).value_counts()

order_item_id
1       509
2       328
3       212
4       152
5       142
       ... 
113       1
209       1
207       1
206       1
2033      1
Name: count, Length: 257, dtype: int64


order_id
1       509
2       328
3       212
4       152
5       142
       ... 
113       1
209       1
207       1
206       1
2033      1
Name: count, Length: 257, dtype: int64

In [25]:
df_ord_itm.groupby('order_id')['price'].count().sort_values()

order_id
00010242fe8c5a6d1ba2dd792cb16214     1
a6e9d106235bcf1dda54253686d89e99     1
a6e9b80a7636eb8dd592dbb3e20d0a91     1
a6e963c11e80432334e984ead4797a8b     1
a6e8ad5db31e71f5f12671af561acb4a     1
                                    ..
428a2f660dc84138d969ccd69a0ab6d5    15
9ef13efd6949e4573a18964dd1bbe7f5    15
1b15974a0141d54e36626dca3fdc731a    20
ab14fdcfbe524636d65ee38360e22ce8    20
8272b63d03f5f79c56e9e4120aec44ef    21
Name: price, Length: 98666, dtype: int64

In [26]:
df_ord_itm.groupby('price')['order_item_id'].count()

price
0.85        3
1.20       20
2.20        1
2.29        1
2.90        1
           ..
4690.00     1
4799.00     1
6499.00     1
6729.00     1
6735.00     1
Name: order_item_id, Length: 5968, dtype: int64

In [27]:
df_ord_itm.groupby('price')['order_id'].count()

price
0.85        3
1.20       20
2.20        1
2.29        1
2.90        1
           ..
4690.00     1
4799.00     1
6499.00     1
6729.00     1
6735.00     1
Name: order_id, Length: 5968, dtype: int64

In [28]:
df_ord_itm['freight_value'].sort_values(ascending=False)

73486     409.68
28044     375.28
3303      375.28
69797     339.59
16731     338.30
           ...  
25980       0.00
23857       0.00
107758      0.00
74348       0.00
39382       0.00
Name: freight_value, Length: 112650, dtype: float64

In [29]:
print(df['olist_order_items_dataset'].shape)
print(df['olist_order_items_dataset'].columns)

(112650, 7)
Index(['order_id', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value'],
      dtype='object')


In [30]:
df['olist_order_items_dataset'].iloc[:,:7].nunique()

order_id               98666
order_item_id             21
product_id             32951
seller_id               3095
shipping_limit_date    93318
price                   5968
freight_value           6999
dtype: int64

>total order rows 112650

>order id 98666

>order item id is 21 so for each item in each order gets number from 1 and so on so order with max products has 21 products

>product id 32951 so these many kind of products are ordered

>rest columns can be ignored as of now

In [31]:
print(df['olist_order_items_dataset'].groupby('order_item_id').size())
df['olist_order_items_dataset'].groupby('order_item_id').size().sum()

order_item_id
1     98666
2      9803
3      2287
4       965
5       460
6       256
7        58
8        36
9        28
10       25
11       17
12       13
13        8
14        7
15        5
16        3
17        3
18        3
19        3
20        3
21        1
dtype: int64


112650

In [32]:
df['olist_order_payments_dataset'].shape

(103886, 5)

In [33]:
df['olist_order_payments_dataset'].iloc[:,:5].nunique()

order_id                99440
payment_sequential         29
payment_type                5
payment_installments       24
payment_value           29077
dtype: int64

In [34]:
olist_cust_db=pd.read_sql_query("select * from olist_customers_dataset",conn)

In [35]:
print(olist_cust_db.shape)
print(olist_cust_db['customer_unique_id'].duplicated().sum())

(99441, 5)
3345


out of 99441 customer dataset only 3345 customers are unique

In [36]:
df.shape

AttributeError: 'dict' object has no attribute 'shape'

In [ ]:
df['geolocation_zip_code_prefix'].nunique()

19015